# Phase 1: Inference & Evaluation Pipeline

In [ ]:
import os
import pickle
import tarfile
import numpy as np
import tensorflow as tf
import sys
import pandas as pd
from pathlib import Path
from mobilenetv2_cifar10_utils import get_mobilenetv2_cifar10
import corruptions

I0000 00:00:1793784711.002939   30826 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793784713.716580   30826 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793784738.936225   30826 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### Dataloader

In [2]:
def load_cifar10_raw(tar_path: str = "cifar-10-python.tar.gz"): #CIFAR-10 from local archive, if needed download with curl -O https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz (or official https://cave.cs.toronto.edu/kriz/cifar-10-python.tar.gz)
    if not os.path.exists(tar_path):
        raise FileNotFoundError(
            f"{tar_path} not found. Place the file in the workspace."
        )

    with tarfile.open(tar_path, "r:gz") as tar:

        def get_batch(name):
            member = [m for m in tar.getmembers() if m.name.endswith(name)][0]
            return pickle.load(tar.extractfile(member), encoding="bytes")

        x_te, y_te = (
            get_batch("test_batch")[b"data"],
            get_batch("test_batch")[b"labels"],
        )

    x_test = x_te.reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
    y_test = np.array(y_te).flatten()
    return x_test, y_test


def get_preprocessed_test_data(tar_path: str = "cifar-10-python.tar.gz", num_samples: int = 500): # test images resized to 224x224
    x_test, y_test = load_cifar10_raw(tar_path)
    x_slice = x_test[:num_samples]
    y_slice = y_test[:num_samples]

    x_resized = tf.image.resize(x_slice, [224, 224]).numpy().astype(np.uint8)
    return x_resized, y_slice

### Generate black-box evaluation records across Clean, Gaussian Noise, Low Contrast, and Fog test sets.

In [ ]:
sys.path.append(str(Path.cwd().parent / "Phase 1 - No AI"))

WEIGHTS_PATH = "../models/float.keras"  # Replace with path to .h5/.keras weights once teammate completes training, this is a placeholder for now
NUM_SAMPLES = 500    # 500 for fast development; 2000-5000 for final runs?
ARCHIVE_PATH = "cifar-10-python.tar.gz"

model, feat_extractor = get_mobilenetv2_cifar10(WEIGHTS_PATH)
images_clean, labels = get_preprocessed_test_data(ARCHIVE_PATH, num_samples=NUM_SAMPLES)
print(f"Loaded {len(images_clean)} test samples.")

/mnt/c/Users/Gabriel/iCloudDrive/JKU/Semester4/AI_for_Visualization/jku-ai-vis-2026/Phase 1 - No AI/mobilenetv2_cifar10_utils.py:16: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights="imagenet", include_top=False, input_tensor=inputs)
E0000 00:00:1793784769.685182   30826 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/mnt/c/Users/Gabriel/iCloudDrive/JKU/Semester4/AI_for_Visualization/jku-ai-vis-2026/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:869: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 0 variables. 
  saveable.load_own_variables(store)


Loaded trained weights from: ../models/float.keras
Loaded 500 test samples.


In [4]:
corruptions_map = {
    "clean": corruptions.apply_clean,
    "gaussian_noise": corruptions.apply_gaussian_noise,
    "low_contrast": corruptions.apply_low_contrast,
    "fog": corruptions.apply_fog,
}

severities = [1, 3, 5]  # sweep across low, medium, and high severity
records = []

for c_name, c_func in corruptions_map.items():
    active_severities = [0] if c_name == "clean" else severities
    for sev in active_severities:
        print(f"Corruption: {c_name} Severity {sev}")
        corrupted_imgs = c_func(images_clean, sev)
        
        # normalized to float32 [0, 255] or preprocess_input
        probs = model.predict(corrupted_imgs.astype(np.float32), batch_size=64, verbose=0) 
        preds = np.argmax(probs, axis=1)
        confs = np.max(probs, axis=1)
        feats = feat_extractor.predict(corrupted_imgs.astype(np.float32), batch_size=64, verbose=0)
        
        for idx in range(len(corrupted_imgs)):
            records.append({
                "sample_id": idx,
                "corruption": c_name,
                "severity": sev,
                "true_label": int(labels[idx]),
                "cnn_pred": int(preds[idx]),
                "cnn_conf": float(confs[idx]),
                "cnn_correct": bool(preds[idx] == labels[idx]),
                "embedding": feats[idx].tolist()
            })

df_results = pd.DataFrame(records)
os.makedirs("../results", exist_ok=True)
output_file = "../results/mobilenetv2_evaluation.parquet"
df_results.to_parquet(output_file, engine="pyarrow")
print(f"Successfully exported {len(df_results)} evaluation rows to {output_file}")

Corruption: clean Severity 0
Corruption: gaussian_noise Severity 1
Corruption: gaussian_noise Severity 3
Corruption: gaussian_noise Severity 5
Corruption: low_contrast Severity 1
Corruption: low_contrast Severity 3
Corruption: low_contrast Severity 5
Corruption: fog Severity 1
Corruption: fog Severity 3
Corruption: fog Severity 5
Successfully exported 5000 evaluation rows to ../results/mobilenetv2_evaluation.parquet
